# Evaluasi Chunking Hybrid vs Recursive

Notebook ini membandingkan collection `COLLECTION_HYBRID_M8_EF400` dan `COLLECTION_RECURSIVE_M8_EF400` untuk query `Q01` dari `Ground truth - System.csv` menggunakan metrik `NDCG@30`.

Evaluasi dilakukan pada ranking artikel hasil retrieval Qdrant. Karena tiap collection berisi chunk, hasil chunk dideduplikasi ke level artikel dengan mengambil chunk terbaik per artikel.

Setiap collection dievaluasi terpisah: targetnya `K=30` untuk hybrid dan `K=30` untuk recursive. Jika `FILTER_TO_GT_POOL=True`, artikel yang tidak ada di ground truth di-skip; karena itu jumlah baris detail bisa kurang dari 30, tetapi perhitungan `NDCG@30` tetap dipad dengan 0 sampai posisi ke-30.

In [8]:
from __future__ import annotations

import importlib
import re
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

try:
    from IPython.display import display
except ImportError:
    display = print

# Path dibuat aman baik notebook dijalankan dari root repo maupun dari folder evaluation.
CWD = Path.cwd().resolve()
if (CWD / "Ground truth - System.csv").exists():
    EVAL_DIR = CWD
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD
    EVAL_DIR = PROJECT_ROOT / "evaluation"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import src.config as config_module

# Reload supaya perubahan config.py terbaca tanpa restart kernel.
config_module = importlib.reload(config_module)
settings = config_module.settings

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EVAL_DIR:", EVAL_DIR)
print("Config file:", config_module.__file__)

PROJECT_ROOT: /home/vinik/Github/vector-search-thesis
EVAL_DIR: /home/vinik/Github/vector-search-thesis/evaluation
Config file: /home/vinik/Github/vector-search-thesis/src/config.py


In [9]:
GT_PATH = EVAL_DIR / "Ground truth - System.csv"
OUTPUT_DIR = EVAL_DIR / "chunk_eval_results"

QUERY_ID = "Q01"
K = 30
SEARCH_HNSW_EF = 256
RAW_QDRANT_LIMIT = 100

# False = artikel yang tidak ada di ground truth Q01 dihitung relevance 0.
# True = hanya artikel yang ada di GT pool yang dihitung.
FILTER_TO_GT_POOL = True

COLLECTION_CONFIGS = [
    {
        "chunking_method": "hybrid",
        "collection_label": "COLLECTION_HYBRID_M8_EF400",
        "collection_name": settings.COLLECTION_HYBRID_M8_EF400,
        "m": 8,
        "ef_construct": 400,
    },
    {
        "chunking_method": "recursive",
        "collection_label": "COLLECTION_RECURSIVE_M8_EF400",
        "collection_name": settings.COLLECTION_RECURSIVE_M8_EF400,
        "m": 8,
        "ef_construct": 400,
    },
]

collection_plan_df = pd.DataFrame(COLLECTION_CONFIGS)
display(collection_plan_df)

,chunking_method,collection_label,collection_name,m,ef_construct
0,hybrid,COLLECTION_HYBRID_M8_EF400,rtas_hybrid_m8_ef400,8,400
1,recursive,COLLECTION_RECURSIVE_M8_EF400,rtas_recursive_m8_ef400,8,400


## Load Ground Truth Q01

In [10]:
def load_ground_truth_for_query(path: Path, query_id: str) -> tuple[pd.DataFrame, str]:
    df = pd.read_csv(path)
    required_columns = {
        "query",
        "query_id",
        "rta_id",
        "agreement_title",
        "chapter_title",
        "article_number",
        "article_title",
        "article_text",
        "relevance",
    }
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing GT columns: {sorted(missing)}")

    query_mask = df["query_id"].astype(str).str.casefold().eq(query_id.casefold())
    query_df = df.loc[query_mask].copy()
    if query_df.empty:
        raise ValueError(f"Tidak ada ground truth untuk query_id={query_id!r}")

    query_df["relevance"] = pd.to_numeric(query_df["relevance"], errors="coerce").fillna(0).astype(float)
    query_values = query_df["query"].dropna().astype(str).unique().tolist()
    if len(query_values) != 1:
        raise ValueError(f"query_id={query_id!r} memiliki lebih dari satu query: {query_values}")

    query_df = query_df.sort_values("relevance", ascending=False).reset_index(drop=True)
    return query_df, query_values[0]


gt_df, QUERY_TEXT = load_ground_truth_for_query(GT_PATH, QUERY_ID)

print(f"Query ID: {QUERY_ID}")
print(f"Query: {QUERY_TEXT}")
print(f"GT rows: {len(gt_df)}")
print("Relevance distribution:")
display(gt_df["relevance"].value_counts().sort_index().rename_axis("relevance").reset_index(name="count"))

display(
    gt_df[
        [
            "query_id",
            "query",
            "rta_id",
            "agreement_title",
            "article_number",
            "article_title",
            "relevance",
        ]
    ].head(10)
)

Query ID: Q01
Query: customs clearance for perishable goods
GT rows: 29
Relevance distribution:


,relevance,count
0,0.0,4
1,1.0,6
2,2.0,9
3,3.0,10


,query_id,query,rta_id,agreement_title,article_number,article_title,relevance
0,Q01,customs clearance for perishable goods,61.0,"China - Hong Kong, China",Article 39,Perishable Goods,3.0
1,Q01,customs clearance for perishable goods,62.0,"China - Macao, China",Article 39,Perishable Goods,3.0
2,Q01,customs clearance for perishable goods,599.0,"Korea, Republic of - Philippines",Article 5.11,Release of Goods,3.0
3,Q01,customs clearance for perishable goods,608.0,United Kingdom - India,Article 5.6,Perishable Goods,3.0
4,Q01,customs clearance for perishable goods,79.0,Chile - China,Article 44,Release of Goods,3.0
5,Q01,customs clearance for perishable goods,164.0,EFTA - Georgia,Article 4,Simplification of International Trade Procedures,3.0
6,Q01,customs clearance for perishable goods,435.0,Regional Comprehensive Economic Partnership Ag...,Article 4.11,Release of Goods,3.0
7,Q01,customs clearance for perishable goods,178.0,"Hong Kong, China - Georgia",Article 13,Perishable Goods,3.0
8,Q01,customs clearance for perishable goods,510.0,United Kingdom - New Zealand FTA,Article 4.10,Perishable Goods,3.0
9,Q01,customs clearance for perishable goods,477.0,United Kingdom - Australia,Article 5.20,Perishable Goodsâ¯,3.0


## Helper Normalisasi dan NDCG

In [11]:
def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = text.replace("\u00e2\u0080\u0099", "'").replace("\u2019", "'").replace("\u00a0", " ")
    text = re.sub(r"\bartical\b", "article", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip().casefold()
    return text


def normalize_rta_id(value: Any) -> str:
    if pd.isna(value) or str(value).strip() == "":
        return ""
    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()


def article_match_keys(
    rta_id: Any,
    agreement_title: Any,
    article_number: Any,
    article_title: Any,
) -> list[tuple[str, ...]]:
    rid = normalize_rta_id(rta_id)
    agreement = normalize_text(agreement_title)
    number = normalize_text(article_number)
    title = normalize_text(article_title)

    keys: list[tuple[str, ...]] = []
    if rid and number and title:
        keys.append(("rta_number_title", rid, number, title))
    if rid and title:
        keys.append(("rta_title", rid, title))
    if agreement and number and title:
        keys.append(("agreement_number_title", agreement, number, title))
    if agreement and title:
        keys.append(("agreement_title", agreement, title))
    return keys


def build_gt_lookup(gt: pd.DataFrame) -> dict[tuple[str, ...], dict[str, Any]]:
    lookup: dict[tuple[str, ...], dict[str, Any]] = {}
    for gt_index, row in gt.iterrows():
        record = {
            "gt_index": gt_index,
            "relevance": float(row["relevance"]),
            "rta_id": row.get("rta_id"),
            "agreement_title": row.get("agreement_title"),
            "chapter_title": row.get("chapter_title"),
            "article_number": row.get("article_number"),
            "article_title": row.get("article_title"),
        }
        for key in article_match_keys(
            row.get("rta_id"),
            row.get("agreement_title"),
            row.get("article_number"),
            row.get("article_title"),
        ):
            if key not in lookup or record["relevance"] > lookup[key]["relevance"]:
                lookup[key] = record
    return lookup


def prediction_match_keys(prediction: dict[str, Any]) -> list[tuple[str, ...]]:
    return article_match_keys(
        prediction.get("rta_id"),
        prediction.get("agreement"),
        prediction.get("article_number"),
        prediction.get("article_name"),
    )


def find_prediction_in_gt(
    prediction: dict[str, Any],
    gt_lookup: dict[tuple[str, ...], dict[str, Any]],
) -> tuple[tuple[str, ...] | None, dict[str, Any] | None]:
    for key in prediction_match_keys(prediction):
        if key in gt_lookup:
            return key, gt_lookup[key]
    return None, None


def compute_dcg(scores: list[float]) -> float:
    # Linear gain, konsisten dengan notebook evaluasi lain di repo ini.
    return sum(float(rel) / np.log2(idx + 2) for idx, rel in enumerate(scores))


def compute_ndcg(predicted_relevance: list[float], ideal_relevance: list[float], k: int = K) -> float:
    scores = list(predicted_relevance[:k])
    if len(scores) < k:
        scores.extend([0.0] * (k - len(scores)))

    ideal = sorted([float(rel) for rel in ideal_relevance], reverse=True)[:k]
    if len(ideal) < k:
        ideal.extend([0.0] * (k - len(ideal)))

    idcg = compute_dcg(ideal)
    return compute_dcg(scores) / idcg if idcg > 0 else 0.0

## Retrieval Qdrant dan Evaluasi Chunking

In [12]:
client = QdrantClient(host=settings.QDRANT_HOST, port=settings.QDRANT_PORT)
vector_model = SentenceTransformer(settings.VECTOR_MODEL)


def resolve_article_text_display(payload: dict[str, Any]) -> str:
    article_text = payload.get("article_text") or "-"
    display_text = payload.get("article_text_display")
    if display_text is None:
        return article_text

    display_text = str(display_text).strip()
    if not display_text or display_text == "-" or display_text.lower().startswith("http") or display_text.lower().startswith("<a"):
        return article_text
    return display_text


def build_prediction_from_hit(hit: Any, rank: int) -> dict[str, Any]:
    payload = hit.payload or {}
    return {
        "rank": rank,
        "point_id": hit.id,
        "score": float(hit.score),
        "rta_id": payload.get("rta_id"),
        "agreement": payload.get("Agreement") or "-",
        "chapter_name": payload.get("chapter_name") or "-",
        "article_id": payload.get("article_id"),
        "article_number": payload.get("article_number") or "-",
        "article_name": payload.get("article_name") or "-",
        "article_text": payload.get("article_text") or "-",
        "article_text_display": resolve_article_text_display(payload),
        "chunk_index": payload.get("chunk_index"),
        "total_chunks": payload.get("total_chunks"),
        "chunk_method": payload.get("chunk_method"),
        "chunk_tokens": payload.get("chunk_tokens"),
    }


def dedupe_hits_to_articles(hits: list[Any]) -> list[Any]:
    unique_hits = []
    seen_article_keys = set()

    for hit in hits:
        payload = hit.payload or {}
        keys = article_match_keys(
            payload.get("rta_id"),
            payload.get("Agreement"),
            payload.get("article_number"),
            payload.get("article_name"),
        )
        dedupe_key = keys[0] if keys else ("point_id", str(hit.id))
        if dedupe_key in seen_article_keys:
            continue
        seen_article_keys.add(dedupe_key)
        unique_hits.append(hit)

    return unique_hits


def retrieve_articles(
    collection_name: str,
    query: str,
    raw_limit: int = RAW_QDRANT_LIMIT,
    hnsw_ef: int = SEARCH_HNSW_EF,
) -> tuple[list[dict[str, Any]], int, float]:
    query_vector = vector_model.encode(query).tolist()

    started = time.perf_counter()
    response = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=raw_limit,
        search_params=models.SearchParams(hnsw_ef=hnsw_ef),
        with_payload=True,
    )
    elapsed_ms = (time.perf_counter() - started) * 1000

    raw_hits = response.points
    unique_hits = dedupe_hits_to_articles(raw_hits)
    predictions = [build_prediction_from_hit(hit, rank=idx) for idx, hit in enumerate(unique_hits, start=1)]
    return predictions, len(raw_hits), elapsed_ms


def evaluate_predictions(
    predictions: list[dict[str, Any]],
    gt: pd.DataFrame,
    k: int = K,
    filter_to_gt_pool: bool = FILTER_TO_GT_POOL,
) -> tuple[float, list[float], list[dict[str, Any]]]:
    gt_lookup = build_gt_lookup(gt)
    relevance_sequence: list[float] = []
    detail_rows: list[dict[str, Any]] = []
    seen_gt_keys = set()

    for prediction in predictions:
        gt_key, gt_record = find_prediction_in_gt(prediction, gt_lookup)

        if gt_record is None:
            if filter_to_gt_pool:
                continue
            relevance = 0.0
        else:
            if gt_key in seen_gt_keys:
                continue
            seen_gt_keys.add(gt_key)
            relevance = float(gt_record["relevance"])

        relevance_sequence.append(relevance)
        detail_rows.append(
            {
                "rank_after_eval_filter": len(relevance_sequence),
                "rank_before_eval_filter": prediction.get("rank"),
                "score": prediction.get("score"),
                "relevance": relevance,
                "matched_in_gt": gt_record is not None,
                "rta_id": prediction.get("rta_id"),
                "agreement": prediction.get("agreement"),
                "chapter_name": prediction.get("chapter_name"),
                "article_number": prediction.get("article_number"),
                "article_name": prediction.get("article_name"),
                "chunk_index": prediction.get("chunk_index"),
                "total_chunks": prediction.get("total_chunks"),
                "chunk_method": prediction.get("chunk_method"),
                "chunk_tokens": prediction.get("chunk_tokens"),
                "gt_rta_id": None if gt_record is None else gt_record.get("rta_id"),
                "gt_agreement_title": None if gt_record is None else gt_record.get("agreement_title"),
                "gt_article_number": None if gt_record is None else gt_record.get("article_number"),
                "gt_article_title": None if gt_record is None else gt_record.get("article_title"),
            }
        )

        if len(relevance_sequence) >= k:
            break

    ndcg = compute_ndcg(relevance_sequence, gt["relevance"].tolist(), k=k)
    return ndcg, relevance_sequence, detail_rows


def evaluate_chunking_collections(
    collection_configs: list[dict[str, Any]],
    gt: pd.DataFrame,
    query: str,
    k: int = K,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    detail_rows = []

    for config in collection_configs:
        collection_name = config["collection_name"]
        print(f"Evaluating {config['collection_label']} ({collection_name})")

        predictions, raw_hit_count, elapsed_ms = retrieve_articles(
            collection_name=collection_name,
            query=query,
            raw_limit=RAW_QDRANT_LIMIT,
            hnsw_ef=SEARCH_HNSW_EF,
        )
        ndcg, relevance_sequence, rows = evaluate_predictions(
            predictions=predictions,
            gt=gt,
            k=k,
            filter_to_gt_pool=FILTER_TO_GT_POOL,
        )

        summary_rows.append(
            {
                "query_id": QUERY_ID,
                "query": query,
                "chunking_method": config["chunking_method"],
                "collection_label": config["collection_label"],
                "collection_name": collection_name,
                "m": config["m"],
                "ef_construct": config["ef_construct"],
                "search_hnsw_ef": SEARCH_HNSW_EF,
                "raw_qdrant_limit": RAW_QDRANT_LIMIT,
                "target_k": k,
                "filter_to_gt_pool": FILTER_TO_GT_POOL,
                "raw_hit_count": raw_hit_count,
                "unique_article_count": len(predictions),
                "evaluated_rows": len(relevance_sequence),
                "matched_gt_rows_at_k": sum(1 for row in rows if row["matched_in_gt"]),
                f"ndcg_at_{k}": ndcg,
                "elapsed_ms": elapsed_ms,
                "relevance_sequence": relevance_sequence,
            }
        )

        for row in rows:
            detail_rows.append(
                {
                    "query_id": QUERY_ID,
                    "query": query,
                    "chunking_method": config["chunking_method"],
                    "collection_label": config["collection_label"],
                    "collection_name": collection_name,
                    **row,
                }
            )

    summary_df = pd.DataFrame(summary_rows).sort_values(f"ndcg_at_{k}", ascending=False).reset_index(drop=True)
    detail_df = pd.DataFrame(detail_rows)
    return summary_df, detail_df

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9056.32it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Jalankan Evaluasi NDCG@30

In [13]:
chunk_summary_df, chunk_detail_df = evaluate_chunking_collections(
    collection_configs=COLLECTION_CONFIGS,
    gt=gt_df,
    query=QUERY_TEXT,
    k=K,
)

summary_cols = [
    "query_id",
    "chunking_method",
    "collection_label",
    "collection_name",
    "search_hnsw_ef",
    "raw_qdrant_limit",
    "target_k",
    "filter_to_gt_pool",
    "raw_hit_count",
    "unique_article_count",
    "evaluated_rows",
    "matched_gt_rows_at_k",
    f"ndcg_at_{K}",
    "elapsed_ms",
    "relevance_sequence",
]

detail_cols = [
    "chunking_method",
    "rank_after_eval_filter",
    "rank_before_eval_filter",
    "score",
    "relevance",
    "matched_in_gt",
    "rta_id",
    "agreement",
    "article_number",
    "article_name",
    "chunk_index",
    "total_chunks",
    "chunk_method",
    "chunk_tokens",
]

display(chunk_summary_df[summary_cols])
if chunk_detail_df.empty:
    display(pd.DataFrame(columns=detail_cols))
else:
    # Preview detail dibuat eksplisit per collection: maksimal K baris hybrid dan K baris recursive.
    detail_preview_df = (
        chunk_detail_df.sort_values(["chunking_method", "rank_after_eval_filter"])
        .groupby("chunking_method", group_keys=False)
        .head(K)
    )
    display(detail_preview_df[detail_cols])

display(
    chunk_detail_df.groupby("chunking_method")
    .size()
    .rename("evaluated_detail_rows")
    .reset_index()
)


Evaluating COLLECTION_HYBRID_M8_EF400 (rtas_hybrid_m8_ef400)
Evaluating COLLECTION_RECURSIVE_M8_EF400 (rtas_recursive_m8_ef400)


,query_id,chunking_method,collection_label,collection_name,search_hnsw_ef,raw_qdrant_limit,raw_hit_count,unique_article_count,evaluated_rows,matched_gt_rows_at_k,ndcg_at_30,elapsed_ms,relevance_sequence
0,Q01,hybrid,COLLECTION_HYBRID_M8_EF400,rtas_hybrid_m8_ef400,256,100,100,97,16,16,0.819576,156.098510,"[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, ..."
1,Q01,recursive,COLLECTION_RECURSIVE_M8_EF400,rtas_recursive_m8_ef400,256,100,100,96,13,13,0.752868,145.839732,"[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 2.0, ..."


,chunking_method,rank_after_eval_filter,rank_before_eval_filter,score,relevance,matched_in_gt,rta_id,agreement,article_number,article_name,chunk_index,total_chunks,chunk_method,chunk_tokens
0,hybrid,1,1,0.824166,3.0,True,61,"China - Hong Kong, China",Article 39,Perishable Goods,1,2,Semantic,103
1,hybrid,2,2,0.821290,3.0,True,62,"China - Macao, China",Article 39,Perishable Goods,1,2,Semantic,97
2,hybrid,3,3,0.816619,3.0,True,79,Chile - China,Article 44,Release of Goods,1,2,Semantic,181
3,hybrid,4,4,0.806459,3.0,True,608,United Kingdom - India,Article 5.6,Perishable Goods,0,1,Semantic,282
4,hybrid,5,5,0.803620,3.0,True,599,"Korea, Republic of - Philippines",Article 5.11,Release of Goods,1,2,Recursive-Fallback,196
5,hybrid,6,7,0.784580,3.0,True,178,"Hong Kong, China - Georgia",Article 13,Perishable Goods,1,2,Semantic,242
6,hybrid,7,8,0.782115,3.0,True,164,EFTA - Georgia,Article 4,Simplification of International Trade Procedures,1,2,Recursive-Fallback,242
7,hybrid,8,9,0.767822,3.0,True,435,Regional Comprehensive Economic Partnership Ag...,Article 4.11,Release of Goods,1,2,Recursive-Fallback,173
8,hybrid,9,10,0.762837,3.0,True,510,United Kingdom - New Zealand FTA,Article 4.10,Perishable Goods,0,1,Semantic,239
9,hybrid,10,30,0.669322,2.0,True,62,"China - Macao, China",Article 38,Release of Goods,1,2,Semantic,128


In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / f"chunking_q{QUERY_ID.lower()}_summary_ndcg_at_{K}.csv"
detail_path = OUTPUT_DIR / f"chunking_q{QUERY_ID.lower()}_detail_ndcg_at_{K}.csv"

chunk_summary_df.to_csv(summary_path, index=False)
chunk_detail_df.to_csv(detail_path, index=False)

print("Saved outputs:")
print("-", summary_path)
print("-", detail_path)

Saved outputs:
- /home/vinik/Github/vector-search-thesis/evaluation/chunk_eval_results/chunking_qq01_summary_ndcg_at_30.csv
- /home/vinik/Github/vector-search-thesis/evaluation/chunk_eval_results/chunking_qq01_detail_ndcg_at_30.csv
